# Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from PIL import Image
import textwrap
from sklearn.manifold import TSNE
from sklearn.preprocessing import MinMaxScaler

# Arborescence

In [ ]:
def afficher_arborescence(root_dir):
    print(f"Structure de : {os.path.abspath(root_dir)}\n")
    
    for root, dirs, files in os.walk(root_dir):
        # Calculer le niveau d'indentation (profondeur)
        level = root.replace(root_dir, '').count(os.sep)
        indent = ' ' * 4 * level
        sub_indent = ' ' * 4 * (level + 1)
        
        # Afficher le nom du dossier actuel
        print(f"{indent}📂 {os.path.basename(root)}/")
        
        # Si le dossier contient des fichiers
        if files:
            # Trier les fichiers par nom pour la cohérence
            files.sort()
            
            # Prendre les 5 premiers
            top_5 = files[:5]
            for f in top_5:
                print(f"{sub_indent}📄 {f}")
            
            # Si plus de 5 fichiers, indiquer le reste
            if len(files) > 5:
                print(f"{sub_indent}... (+ {len(files) - 5} autres fichiers)")

# Définition du chemin vers ton dossier
path_tfe_data = os.path.expanduser("~/tfe/TFE_Data")

# Exécution
if os.path.exists(path_tfe_data):
    afficher_arborescence(path_tfe_data)
else:
    print(f"Erreur : Le dossier {path_tfe_data} n'existe pas.")

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from PIL import Image

def generate_full_report(base_dir, stress_test_indices, datasets, vision_models, text_models):
    """
    Génère une visualisation comparative pour chaque image du stress test, 
    pour chaque modèle et chaque dataset.
    """
    for ds in datasets:
        print(f"Processing Dataset: {ds}")
        # Charger le dataframe une seule fois par dataset
        df_path = os.path.join(base_dir, "Datasets", f"df_{ds}.pkl")
        df = pd.read_pickle(df_path)
        
        results_ds_dir = os.path.join(base_dir, "Results", ds)

        for img_idx in stress_test_indices:
            # Récupérer l'image et la légende
            img_row = df.iloc[img_idx]
            img_path = img_row['image_path']
            # On prend la première légende si c'est une liste
            caption = img_row['captions'][0] if isinstance(img_row['captions'], list) else img_row['captions']
            
            if not os.path.exists(img_path):
                continue
            
            img_org = Image.open(img_path)

            # Créer une figure large : Vision à gauche, Texte à droite
            # Nombre de lignes = max(nb_modeles_vision, nb_modeles_texte)
            n_rows = max(len(vision_models), len(text_models))
            fig, axes = plt.subplots(n_rows, 2, figsize=(15, 5 * n_rows))
            
            # --- PARTIE VISION (Colonne 0) ---
            for i, v_mod in enumerate(vision_models):
                ax = axes[i, 0]
                v_path = os.path.join(results_ds_dir, "vision_xai", f"SaliencyMap_{v_mod}_{img_idx}.npy")
                # Essayer l'autre nommage si le premier échoue
                if not os.path.exists(v_path):
                    v_path = os.path.join(results_ds_dir, "vision_xai", f"Saliency_{v_mod}_{img_idx}.npy")
                
                ax.imshow(img_org)
                if os.path.exists(v_path):
                    v_sal = np.load(v_path)
                    v_sal_res = cv2.resize(v_sal, (img_org.size[0], img_org.size[1]))
                    ax.imshow(v_sal_res, cmap='jet', alpha=0.4)
                    ax.set_title(f"Vision Model: {v_mod.upper()}")
                else:
                    ax.set_title(f"{v_mod.upper()} (NOT FOUND)")
                ax.axis('off')

            # --- PARTIE TEXTE (Colonne 1) ---
            for i, t_mod in enumerate(text_models):
                ax = axes[i, 1]
                t_path = os.path.join(results_ds_dir, "text_xai", f"SaliencyText_{t_mod}_{img_idx}.npy")
                
                if os.path.exists(t_path):
                    t_data = np.load(t_path, allow_pickle=True).item()
                    tokens, scores = t_data['tokens'], t_data['scores']
                    
                    # Nettoyage rapide pour l'affichage
                    words = [t.replace('##', '').replace('Ġ', '') for t in tokens if t not in ['[CLS]', '[SEP]', '<s>', '</s>', '<pad>']]
                    weights = scores[:len(words)] # Aligner les tailles
                    
                    ax.barh(words[:15], weights[:15], color='salmon') # Limiter aux 15 premiers tokens
                    ax.invert_yaxis()
                    ax.set_title(f"Text Model: {t_mod.upper()}")
                else:
                    ax.set_title(f"{t_mod.upper()} (NOT FOUND)")
            
            plt.suptitle(f"STRESS TEST - ID: {img_idx}\nCaption: {caption}", fontsize=16, fontweight='bold')
            plt.tight_layout(rect=[0, 0.03, 1, 0.95])
            
            # Optionnel : Sauvegarder automatiquement le comparatif
            save_path = os.path.join(results_ds_dir, f"Comparison_ID_{img_idx}.png")
            plt.savefig(save_path)
            plt.show()

# --- CONFIGURATION FINALE ---
BASE_PATH = "/home/aysel/tfe/TFE_Data"
STRESS_TEST = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9] # Tes 10 images
DS_LIST = ['Flickr8k', 'Flickr30k', 'ConceptualCaptions']
MODELS_V = ['resnet50', 'deit', 'clip_vision', 'vit', 'mobilenet_v3']
MODELS_T = ['bert', 'roberta', 'gpt2', 'clip_text', 'rnn']

generate_full_report(BASE_PATH, STRESS_TEST, DS_LIST, MODELS_V, MODELS_T)

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from PIL import Image

def generate_fixed_paths_report(base_dir, datasets, vision_models, text_models, n_images=5):
    for ds in datasets:
        print(f"\n--- 📂 Processing Dataset: {ds} ---")
        
        ds_results_dir = os.path.join(base_dir, "Results", ds)
        v_saliency_folder = os.path.join(ds_results_dir, f"SaliencyMaps_{ds}")
        t_saliency_folder = os.path.join(ds_results_dir, f"TextSaliency_{ds}")
        
        # 1. Charger le DF
        df_path = os.path.join(base_dir, "Datasets", f"df_{ds}.pkl")
        if not os.path.exists(df_path): continue
        df = pd.read_pickle(df_path)

        # 2. Définir le dossier IMAGE réel selon le dataset
        if ds == 'Flickr8k':
            img_base_folder = os.path.join(base_dir, "Datasets", "Flickr8k", "Images", "Flicker8k_Dataset")
        else:
            img_base_folder = os.path.join(base_dir, "Datasets", ds, "Images")

        for img_idx in range(n_images):
            try:
                img_row = df.iloc[img_idx]
                # On récupère juste le nom du fichier (ex: 123.jpg) et on le colle au bon dossier
                img_filename = os.path.basename(img_row['image_path'])
                real_img_path = os.path.join(img_base_folder, img_filename)
                
                caption = img_row['captions'][0] if isinstance(img_row['captions'], list) else img_row['captions']
                
                if not os.path.exists(real_img_path):
                    print(f"⚠️ Toujours introuvable: {real_img_path}")
                    continue

                img_org = Image.open(real_img_path)
                n_rows = max(len(vision_models), len(text_models))
                fig, axes = plt.subplots(n_rows, 2, figsize=(16, 4 * n_rows))

                # --- VISION ---
                for i, v_mod in enumerate(vision_models):
                    ax = axes[i, 0]
                    v_file = os.path.join(v_saliency_folder, f"Saliency_{v_mod}_{img_idx}.npy")
                    ax.imshow(img_org)
                    if os.path.exists(v_file):
                        v_sal = np.load(v_file)
                        v_sal_res = cv2.resize(v_sal, (img_org.size[0], img_org.size[1]))
                        ax.imshow(v_sal_res, cmap='jet', alpha=0.4)
                        ax.set_title(f"Vision: {v_mod.upper()}")
                    ax.axis('off')

                # --- TEXTE ---
                for i, t_mod in enumerate(text_models):
                    ax = axes[i, 1]
                    t_file = os.path.join(t_saliency_folder, f"SaliencyText_{t_mod}_{img_idx}.npy")
                    if os.path.exists(t_file):
                        t_data = np.load(t_file, allow_pickle=True).item()
                        words = [t.replace('##', '').replace('Ġ', '') for t in t_data['tokens'] 
                                 if t not in ['[CLS]', '[SEP]', '<s>', '</s>', '<pad>']]
                        weights = t_data['scores'][:len(words)]
                        ax.barh(words[:15], weights[:15], color='skyblue')
                        ax.invert_yaxis()
                        ax.set_title(f"Text: {t_mod.upper()}")

                plt.suptitle(f"MULTIMODAL CHECK - {ds} | ID: {img_idx}\n{caption[:100]}...", fontsize=14)
                plt.tight_layout(rect=[0, 0.03, 1, 0.95])
                
                save_name = os.path.join(ds_results_dir, f"SanityCheck_ID_{img_idx}.png")
                plt.savefig(save_name)
                plt.close()
                print(f"✅ Succès pour ID {img_idx}")

            except Exception as e:
                print(f"❌ Erreur ID {img_idx}: {e}")

# Lancement
generate_fixed_paths_report(BASE_PATH, DATASETS, MODELS_V, MODELS_T, n_images=5)

# Indexation

In [ ]:
def generate_full_5_caption_report(base_directory, datasets, vision_models, text_models, num_images=3):
    """
    Generates a high-density 5x6 grid report.
    Column 1: Original Image
    Columns 2-6: NLP Saliency for Captions 1 through 5.
    Vision Saliency is overlaid on Column 1 or shown as a separate reference.
    """
    for dataset in datasets:
        print(f"Processing Dataset: {dataset}")
        
        results_path = os.path.join(base_directory, "Results", dataset)
        vision_saliency_path = os.path.join(results_path, f"SaliencyMaps_{dataset}")
        text_saliency_path = os.path.join(results_path, f"TextSaliency_{dataset}")
        
        output_dir = os.path.join(results_path, "Full_5_Caption_Reports")
        os.makedirs(output_dir, exist_ok=True)
        
        df_path = os.path.join(base_directory, "Datasets", f"df_{dataset}.pkl")
        if not os.path.exists(df_path):
            continue
        dataframe = pd.read_pickle(df_path)

        # Dataset-specific image folders
        if dataset == 'Flickr8k':
            img_source_dir = os.path.join(base_directory, "Datasets", "Flickr8k", "Images", "Flicker8k_Dataset")
            max_captions = 5
        elif dataset == 'Flickr30k':
            img_source_dir = os.path.join(base_directory, "Datasets", dataset, "Images")
            max_captions = 5
        else:
            img_source_dir = os.path.join(base_directory, "Datasets", dataset, "Images")
            max_captions = 1

        for img_idx in range(num_images):
            try:
                row = dataframe.iloc[img_idx]
                file_full_name = os.path.basename(row['image_path'])
                file_id = os.path.splitext(file_full_name)[0]
                absolute_img_path = os.path.join(img_source_dir, file_full_name)
                
                if not os.path.exists(absolute_img_path):
                    continue
                    
                raw_image = Image.open(absolute_img_path)
                image_width, image_height = raw_image.size
                captions_list = row['captions'] if isinstance(row['captions'], list) else [row['captions']]

                # Grid: 5 Rows (Models) x 6 Columns (1 Image + 5 Captions)
                fig, axes = plt.subplots(5, 6, figsize=(35, 25))

                for row_idx in range(5):
                    v_model = vision_models[row_idx]
                    t_model = text_models[row_idx]

                    # COLUMN 0: Vision Saliency (Reference)
                    ax_vis = axes[row_idx, 0]
                    v_file = os.path.join(vision_saliency_path, f"Saliency_{v_model}_{img_idx}.npy")
                    ax_vis.imshow(raw_image)
                    if os.path.exists(v_file):
                        heatmap = np.load(v_file)
                        heatmap_resized = cv2.resize(heatmap, (image_width, image_height))
                        ax_vis.imshow(heatmap_resized, cmap='jet', alpha=0.4)
                    ax_vis.set_ylabel(f"ARCH: {v_model.upper()} / {t_model.upper()}", 
                                     fontsize=14, fontweight='bold')
                    if row_idx == 0:
                        ax_vis.set_title("VISION ALIGNMENT", fontweight='bold', fontsize=15)
                    ax_vis.set_xticks([]); ax_vis.set_yticks([])

                    # COLUMNS 1 to 5: Text Saliency for each Caption
                    for cap_idx in range(5):
                        ax_txt = axes[row_idx, cap_idx + 1]
                        
                        if cap_idx < len(captions_list):
                            # Global index calculation
                            global_text_idx = (img_idx * 5) + cap_idx
                            t_file = os.path.join(text_saliency_path, f"SaliencyText_{t_model}_{global_text_idx}.npy")
                            
                            if os.path.exists(t_file):
                                data_struct = np.load(t_file, allow_pickle=True).item()
                                excluded = ['[CLS]', '[SEP]', '<s>', '</s>', '<pad>', '<|endoftext|>']
                                tokens = [t.replace('##', '').replace('Ġ', '') for t in data_struct['tokens'] 
                                         if t not in excluded]
                                scores = data_struct['scores'][:len(tokens)]
                                
                                ax_txt.barh(tokens[:12], scores[:12], color='darkcyan')
                                ax_txt.invert_yaxis()
                                
                                # Add truncated caption text as subtitle for each bar chart
                                short_cap = "\n".join(textwrap.wrap(captions_list[cap_idx], width=25))
                                ax_txt.set_xlabel(short_cap, fontsize=9, fontstyle='italic')
                            else:
                                ax_txt.text(0.5, 0.5, "NPY Missing", ha='center')
                        else:
                            ax_txt.axis('off')
                        
                        if row_idx == 0:
                            ax_txt.set_title(f"CAPTION {cap_idx+1}", fontweight='bold', fontsize=15)

                plt.suptitle(f"Multi-Caption Semantic Interpretability Analysis\nDataset: {dataset} | Image: {file_full_name}", 
                             fontsize=22, fontweight='bold', y=0.98)
                
                plt.tight_layout(rect=[0, 0.05, 1, 0.95])
                save_path = os.path.join(output_dir, f"Full_Analysis_{file_id}.png")
                plt.savefig(save_path, dpi=120, bbox_inches='tight')
                plt.close()
                print(f"Exported High-Density Report: {file_full_name}")

            except Exception as e:
                print(f"Failed processing image {img_idx}: {str(e)}")

if __name__ == "__main__":
    BASE_PATH = "/home/aysel/tfe/TFE_Data"
    DATASETS = ['Flickr8k', 'Flickr30k'] # Restricted to multi-caption datasets
    VISION_ORDER = ['resnet50', 'mobilenet_v3', 'vit', 'deit', 'clip_vision']
    TEXT_ORDER = ['bert', 'roberta', 'rnn', 'gpt2', 'clip_text']
    
    generate_full_5_caption_report(BASE_PATH, DATASETS, VISION_ORDER, TEXT_ORDER)

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import textwrap
from PIL import Image
import cv2

def clean_and_normalize_text_data(data, max_tokens=15):
    """Nettoie les tokens et force la normalisation des scores (Crucial pour RNN)."""
    if not isinstance(data, dict): return [], []
    excluded = ['[CLS]', '[SEP]', '<s>', '</s>', '<pad>', '<|endoftext|>', '', ' ']
    
    raw_tokens = data.get('tokens', [])
    raw_scores = data.get('scores', [])
    
    clean_tokens, clean_scores = [], []
    for t, s in zip(raw_tokens, raw_scores):
        t_clean = str(t).replace('##', '').replace('Ġ', '').strip()
        if t_clean and t_clean not in excluded:
            clean_tokens.append(t_clean)
            clean_scores.append(float(s) if np.isfinite(float(s)) else 0.0)
            
    if not clean_tokens: return [], []

    # Normalisation Robuste pour que les barres soient visibles
    scores_np = np.array(clean_scores)
    s_min, s_max = scores_np.min(), scores_np.max()
    if s_max > s_min:
        scores_np = (scores_np - s_min) / (s_max - s_min)
    else:
        scores_np = np.full_like(scores_np, 0.5)

    return clean_tokens[:max_tokens], scores_np[:max_tokens].tolist()

def generate_full_5_caption_report(base_directory, datasets, vision_models, text_models, num_images=5):
    for dataset in datasets:
        print(f"\n--- Processing Dataset: {dataset} ---")
        
        # Chemins selon votre structure
        df_path = os.path.join(base_directory, "Datasets", f"df_{dataset}.pkl")
        if not os.path.exists(df_path): continue
        dataframe = pd.read_pickle(df_path)

        # Gestion des dossiers images spécifiques
        if dataset == 'Flickr8k':
            img_source_dir = os.path.join(base_directory, "Datasets", "Flickr8k", "Images", "Flicker8k_Dataset")
        elif dataset == 'Flickr30k':
            img_source_dir = os.path.join(base_directory, "Datasets", "Flickr30k", "Images")
        else:
            img_source_dir = os.path.join(base_directory, "Datasets", dataset, "Images")

        output_dir = os.path.join(base_directory, "Results", dataset, "Sanity_Checks", "Full_5_Caption_Reports")
        os.makedirs(output_dir, exist_ok=True)

        for img_idx in range(min(num_images, len(dataframe))):
            try:
                row = dataframe.iloc[img_idx]
                file_full_name = os.path.basename(row['image_path'])
                file_id = os.path.splitext(file_full_name)[0]
                absolute_img_path = os.path.join(img_source_dir, file_full_name)
                
                if not os.path.exists(absolute_img_path): continue
                raw_image = Image.open(absolute_img_path)
                w, h = raw_image.size
                captions_list = row['captions'] if isinstance(row['captions'], list) else [row['captions']]

                fig, axes = plt.subplots(5, 6, figsize=(35, 25))

                for row_idx in range(5):
                    v_model = vision_models[row_idx]
                    t_model = text_models[row_idx]

                    # 1. VISION (Col 0) - Chemin: Results/Dataset/Model/SaliencyMap/
                    ax_vis = axes[row_idx, 0]
                    v_file = os.path.join(base_directory, "Results", dataset, v_model, "SaliencyMap", f"Saliency_{v_model}_{img_idx}.npy")
                    
                    ax_vis.imshow(raw_image)
                    if os.path.exists(v_file):
                        heatmap = cv2.resize(np.load(v_file), (w, h))
                        ax_vis.imshow(heatmap, cmap='jet', alpha=0.4)
                    
                    ax_vis.set_ylabel(f"{v_model.upper()} / {t_model.upper()}", fontsize=14, fontweight='bold')
                    ax_vis.set_xticks([]); ax_vis.set_yticks([])

                    # 2. TEXT (Cols 1-5) - Chemin: Results/Dataset/Model/SaliencyMap/
                    for cap_idx in range(5):
                        ax_txt = axes[row_idx, cap_idx + 1]
                        if cap_idx < len(captions_list):
                            global_idx = (img_idx * 5) + cap_idx
                            t_file = os.path.join(base_directory, "Results", dataset, t_model, "SaliencyMap", f"SaliencyText_{t_model}_{global_idx}.npy")
                            
                            if os.path.exists(t_file):
                                data = np.load(t_file, allow_pickle=True).item()
                                tokens, scores = clean_and_normalize_text_data(data, 12)
                                if tokens:
                                    ax_txt.barh(tokens, scores, color='darkcyan', edgecolor='black')
                                    ax_txt.invert_yaxis()
                                    wrapped_cap = "\n".join(textwrap.wrap(captions_list[cap_idx], width=25))
                                    ax_txt.set_xlabel(wrapped_cap, fontsize=10, fontstyle='italic')
                                else: ax_txt.text(0.5, 0.5, "Empty Data", ha='center')
                            else: ax_txt.text(0.5, 0.5, "File Missing", ha='center')
                        else: ax_txt.axis('off')

                plt.suptitle(f"Multi-Caption Analysis | {dataset} | {file_full_name}", fontsize=25, fontweight='bold', y=0.98)
                plt.tight_layout(rect=[0, 0.05, 1, 0.95])
                plt.savefig(os.path.join(output_dir, f"Full_Analysis_{file_id}.png"), dpi=100)
                plt.close()
                print(f"✓ Saved: {file_id}")

            except Exception as e:
                print(f"✗ Error image {img_idx}: {e}")

if __name__ == "__main__":
    BASE = "/home/aysel/tfe/TFE_Data"
    # Ordre strict correspondant à vos dossiers
    V_MODELS = ['resnet50', 'mobilenet_v3', 'vit', 'deit', 'clip_vision']
    T_MODELS = ['bert', 'roberta', 'rnn', 'gpt2', 'clip_text']
    
    generate_full_5_caption_report(BASE, ['Flickr8k', 'Flickr30k'], V_MODELS, T_MODELS, num_images=5)

In [ ]:
def generate_consensus_report(base_directory, datasets, vision_models, text_models, num_images=3):
    """
    Generates a report showing the overlap of token importance across 5 captions.
    """
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd'] # Distinct colors for 5 captions

    for dataset in datasets:
        if dataset not in ['Flickr8k', 'Flickr30k']: continue
        
        results_path = os.path.join(base_directory, "Results", dataset)
        text_saliency_path = os.path.join(results_path, f"TextSaliency_{dataset}")
        output_dir = os.path.join(results_path, "Consensus_Reports")
        os.makedirs(output_dir, exist_ok=True)
        
        dataframe = pd.read_pickle(os.path.join(base_directory, "Datasets", f"df_{dataset}.pkl"))
        img_source_dir = os.path.join(base_directory, "Datasets", dataset, "Images")
        if dataset == 'Flickr8k':
            img_source_dir = os.path.join(img_source_dir, "Flicker8k_Dataset")

        for img_idx in range(num_images):
            row = dataframe.iloc[img_idx]
            file_name = os.path.basename(row['image_path'])
            captions = row['captions']
            raw_image = Image.open(os.path.join(img_source_dir, file_name))

            fig, axes = plt.subplots(5, 2, figsize=(20, 30))
            
            for m_idx, t_model in enumerate(text_models):
                # Col 1: Image + Text Block
                ax_img = axes[m_idx, 0]
                ax_img.imshow(raw_image)
                if m_idx == 0:
                    text_block = "\n".join([f"{i+1}: {textwrap.shorten(c, width=50)}" for i, c in enumerate(captions)])
                    ax_img.text(0, -0.1, text_block, transform=ax_img.transAxes, va='top', fontsize=9, family='monospace')
                ax_img.set_ylabel(t_model.upper(), fontweight='bold', fontsize=14)
                ax_img.set_xticks([]); ax_img.set_yticks([])

                # Col 2: Overlap Chart
                ax_ovl = axes[m_idx, 1]
                all_tokens = []
                all_scores = []
                token_colors = []

                for cap_idx in range(5):
                    global_idx = (img_idx * 5) + cap_idx
                    t_file = os.path.join(text_saliency_path, f"SaliencyText_{t_model}_{global_idx}.npy")
                    
                    if os.path.exists(t_file):
                        data = np.load(t_file, allow_pickle=True).item()
                        excluded = ['[CLS]', '[SEP]', '<s>', '</s>', '<pad>', '<|endoftext|>']
                        for t, s in zip(data['tokens'], data['scores']):
                            clean_t = t.replace('##', '').replace('Ġ', '')
                            if clean_t not in excluded and len(clean_t) > 1:
                                all_tokens.append(f"{clean_t} (C{cap_idx+1})")
                                all_scores.append(s)
                                token_colors.append(colors[cap_idx])

                # Plot top 25 tokens across all captions to see the dominant ones
                indices = np.argsort(all_scores)[-25:]
                ax_ovl.barh([all_tokens[i] for i in indices], [all_scores[i] for i in indices], 
                            color=[token_colors[i] for i in indices])
                ax_ovl.set_title(f"{t_model.upper()}: Top Tokens Cross-Caption Overlap", fontweight='bold')

            plt.suptitle(f"Semantic Consensus Analysis: {file_name}", fontsize=20, fontweight='bold', y=0.98)
            plt.tight_layout(rect=[0, 0.05, 1, 0.96])
            plt.savefig(os.path.join(output_dir, f"Consensus_{file_name}.png"), dpi=150)
            plt.close()

generate_consensus_report("/home/aysel/tfe/TFE_Data", ['Flickr8k'], [], ['bert', 'roberta', 'rnn', 'gpt2', 'clip_text'])

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import textwrap
from PIL import Image
import cv2

def get_image_source_path(base_directory, dataset):
    """Chemin vers les images source par dataset."""
    if dataset == 'Flickr8k':
        return os.path.join(base_directory, "Datasets", "Flickr8k", "Images", "Flicker8k_Dataset")
    return os.path.join(base_directory, "Datasets", dataset, "Images")

def clean_and_normalize_text_data(data, max_tokens=15):
    """
    Nettoyage strict et normalisation pour forcer l'affichage,
    particulièrement pour les modèles RNN à faibles scores.
    """
    if not isinstance(data, dict):
        return [], []
        
    excluded = ['[CLS]', '[SEP]', '<s>', '</s>', '<pad>', '<|endoftext|>', '', ' ']
    raw_tokens = data.get('tokens', [])
    raw_scores = data.get('scores', [])
    
    clean_tokens, clean_scores = [], []
    
    # Synchronisation et nettoyage
    for t, s in zip(raw_tokens, raw_scores):
        t_clean = str(t).replace('##', '').replace('Ġ', '').strip()
        if t_clean and t_clean not in excluded:
            clean_tokens.append(t_clean)
            # Gestion des NaN ou Inf dans les scores
            val = float(s) if np.isfinite(float(s)) else 0.0
            clean_scores.append(val)
            
    if not clean_tokens:
        return [], []

    # Normalisation Robuste (0 à 1)
    scores_np = np.array(clean_scores)
    s_min, s_max = scores_np.min(), scores_np.max()
    
    if s_max > s_min:
        scores_np = (scores_np - s_min) / (s_max - s_min)
    else:
        # Si tous les scores sont identiques (ex: tous à 0), on met 0.5 pour la visibilité
        scores_np = np.full_like(scores_np, 0.5)

    # On s'assure qu'on ne renvoie pas de listes vides pour le plot
    final_tokens = clean_tokens[:max_tokens]
    final_scores = scores_np[:max_tokens].tolist()
    
    return final_tokens, final_scores

def generate_multimodal_alignment_report(base_directory, datasets, vision_models, text_models, num_samples=5):
    """Génère les rapports d'analyse pour tous les modèles et datasets."""
    for dataset in datasets:
        print(f"Generating Multimodal Report: {dataset}")
        output_dir = os.path.join(base_directory, "Results", dataset, "Sanity_Checks", "Multimodal_Alignment_Reports")
        os.makedirs(output_dir, exist_ok=True)
        
        df_path = os.path.join(base_directory, "Datasets", f"df_{dataset}.pkl")
        if not os.path.exists(df_path): continue
        dataframe = pd.read_pickle(df_path)
        img_source_dir = get_image_source_path(base_directory, dataset)

        for idx in range(min(num_samples, len(dataframe))):
            try:
                row = dataframe.iloc[idx]
                file_name = os.path.basename(row['image_path'])
                img_path = os.path.join(img_source_dir, file_name)
                if not os.path.exists(img_path): continue
                
                raw_img = Image.open(img_path)
                w, h = raw_img.size
                caption = row['captions'][0] if isinstance(row['captions'], list) else row['captions']

                fig, axes = plt.subplots(5, 3, figsize=(20, 25))
                for i in range(5):
                    # Image de référence (Col 1)
                    axes[i, 0].imshow(raw_img)
                    if i == 0:
                        wrapped = "\n".join(textwrap.wrap(f"Source: {caption}", width=40))
                        axes[i, 0].set_title("REFERENCE IMAGE", fontweight='bold', fontsize=14)
                        axes[i, 0].text(0.5, -0.2, wrapped, ha='center', transform=axes[i, 0].transAxes)
                    axes[i, 0].axis('off')

                    # Vision Saliency (Col 2)
                    v_m = vision_models[i]
                    v_p = os.path.join(base_directory, "Results", dataset, v_m, "SaliencyMap", f"Saliency_{v_m}_{idx}.npy")
                    axes[i, 1].imshow(raw_img)
                    if os.path.exists(v_p):
                        hm = cv2.resize(np.load(v_p), (w, h))
                        axes[i, 1].imshow(hm, cmap='jet', alpha=0.4)
                        axes[i, 1].set_title(f"VISION: {v_m.upper()}", fontweight='bold')
                    axes[i, 1].axis('off')

                    # Text Saliency (Col 3) - Fix RNN
                    t_m = text_models[i]
                    t_p = os.path.join(base_directory, "Results", dataset, t_m, "SaliencyMap", f"SaliencyText_{t_m}_{idx}.npy")
                    if os.path.exists(t_p):
                        data_load = np.load(t_p, allow_pickle=True).item()
                        t_toks, t_scs = clean_and_normalize_text_data(data_load)
                        if t_toks:
                            axes[i, 2].barh(t_toks, t_scs, color='steelblue', edgecolor='black')
                            axes[i, 2].invert_yaxis()
                            axes[i, 2].set_title(f"TEXT: {t_m.upper()}", fontweight='bold')
                        else: axes[i, 2].text(0.5, 0.5, "No Data", ha='center')
                    else: axes[i, 2].text(0.5, 0.5, "File Missing", ha='center')

                plt.suptitle(f"Analysis Report - Dataset: {dataset} - Image: {file_name}", fontsize=20, y=0.98, fontweight='bold')
                plt.tight_layout(rect=[0, 0.03, 1, 0.96])
                plt.savefig(os.path.join(output_dir, f"Analysis_{os.path.splitext(file_name)[0]}.png"), dpi=120)
                plt.close()
            except Exception as e: print(f"Error Multimodal {idx}: {e}")

# ... (Les autres fonctions generate_full_5_caption_report et generate_consensus_report 
# bénéficient automatiquement de la correction car elles utilisent clean_and_normalize_text_data)

if __name__ == "__main__":
    PATH = "/home/aysel/tfe/TFE_Data"
    DB = ['Flickr8k', 'Flickr30k', 'ConceptualCaptions']
    VM = ['resnet50', 'mobilenet_v3', 'vit', 'deit', 'clip_vision']
    TM = ['bert', 'roberta', 'rnn', 'gpt2', 'clip_text']
    
    # Exécution sur les 5 premières images de chaque dataset pour tous les rapports
    generate_multimodal_alignment_report(PATH, DB, VM, TM, 5)
    # Note : Les fonctions suivantes ne tournent que sur Flickr car CC n'a qu'une seule caption
    from __main__ import generate_full_5_caption_report, generate_consensus_report # facultatif si dans le même fichier
    generate_full_5_caption_report(PATH, ['Flickr8k', 'Flickr30k'], VM, TM, 5)
    generate_consensus_report(PATH, ['Flickr8k', 'Flickr30k'], TM, 5)

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import textwrap
from PIL import Image
import cv2

def clean_and_normalize_text_data(data, max_tokens=15):
    """Nettoyage et normalisation forcée pour la visibilité (Spécial RNN)."""
    if not isinstance(data, dict): return [], []
    excluded = ['[CLS]', '[SEP]', '<s>', '</s>', '<pad>', '<|endoftext|>', '', ' ']
    
    raw_tokens = data.get('tokens', [])
    raw_scores = data.get('scores', [])
    
    clean_tokens, clean_scores = [], []
    for t, s in zip(raw_tokens, raw_scores):
        t_clean = str(t).replace('##', '').replace('Ġ', '').strip()
        if t_clean and t_clean not in excluded:
            clean_tokens.append(t_clean)
            clean_scores.append(float(s) if np.isfinite(float(s)) else 0.0)
            
    if not clean_tokens: return [], []

    # Normalisation 0-1 pour éviter les barres invisibles
    scores_np = np.array(clean_scores)
    if scores_np.max() > scores_np.min():
        scores_np = (scores_np - scores_np.min()) / (scores_np.max() - scores_np.min())
    else:
        scores_np = np.full_like(scores_np, 0.5)

    return clean_tokens[:max_tokens], scores_np[:max_tokens].tolist()


def generate_full_5_caption_report(base_directory, datasets, vision_models, text_models, num_images=5):
    """Rapport 5x6 haute densité (Flickr uniquement)."""
    for dataset in datasets:
        if dataset not in ['Flickr8k', 'Flickr30k']: continue
        print(f"Génération Full 5-Captions : {dataset}")
        output_dir = os.path.join(base_directory, "Results", dataset, "Sanity_Checks", "Full_5_Caption_Reports")
        os.makedirs(output_dir, exist_ok=True)
        
        df = pd.read_pickle(os.path.join(base_directory, "Datasets", f"df_{dataset}.pkl"))
        img_dir = os.path.join(base_directory, "Datasets", dataset, "Images", "Flicker8k_Dataset" if dataset == 'Flickr8k' else "")

        for img_idx in range(num_images):
            row = df.iloc[img_idx]
            img_name = os.path.basename(row['image_path'])
            raw_img = Image.open(os.path.join(img_dir, img_name))
            w, h = raw_img.size
            
            fig, axes = plt.subplots(5, 6, figsize=(35, 25))
            for r in range(5):
                vm, tm = vision_models[r], text_models[r]
                # Vision (Col 0)
                v_p = os.path.join(base_directory, "Results", dataset, vm, "SaliencyMap", f"Saliency_{vm}_{img_idx}.npy")
                axes[r, 0].imshow(raw_img)
                if os.path.exists(v_p):
                    axes[r, 0].imshow(cv2.resize(np.load(v_p), (w, h)), cmap='jet', alpha=0.4)
                axes[r, 0].set_ylabel(f"{vm.upper()}/{tm.upper()}", fontweight='bold')
                
                # Text (Cols 1-5)
                for c in range(5):
                    t_p = os.path.join(base_directory, "Results", dataset, tm, "SaliencyMap", f"SaliencyText_{tm}_{(img_idx*5)+c}.npy")
                    if os.path.exists(t_p):
                        toks, scs = clean_and_normalize_text_data(np.load(t_p, allow_pickle=True).item(), 12)
                        if toks:
                            axes[r, c+1].barh(toks, scs, color='darkcyan')
                            axes[r, c+1].invert_yaxis()
                            axes[r, c+1].set_xlabel("\n".join(textwrap.wrap(row['captions'][c], 25)), fontsize=8)
            plt.savefig(os.path.join(output_dir, f"Full_{os.path.splitext(img_name)[0]}.png"))
            plt.close()

def generate_consensus_report(base_directory, datasets, text_models, num_images=5):
    """Rapport de consensus sémantique."""
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']
    for dataset in datasets:
        if dataset not in ['Flickr8k', 'Flickr30k']: continue
        print(f"Génération Consensus : {dataset}")
        output_dir = os.path.join(base_directory, "Results", dataset, "Sanity_Checks", "Consensus_Reports")
        os.makedirs(output_dir, exist_ok=True)
        
        df = pd.read_pickle(os.path.join(base_directory, "Datasets", f"df_{dataset}.pkl"))
        img_dir = os.path.join(base_directory, "Datasets", dataset, "Images", "Flicker8k_Dataset" if dataset == 'Flickr8k' else "")

        for img_idx in range(num_images):
            row = df.iloc[img_idx]
            img_name = os.path.basename(row['image_path'])
            fig, axes = plt.subplots(5, 2, figsize=(20, 30))
            for m_idx, tm in enumerate(text_models):
                axes[m_idx, 0].imshow(Image.open(os.path.join(img_dir, img_name)))
                axes[m_idx, 0].set_ylabel(tm.upper(), fontweight='bold')
                
                all_t, all_s, all_c = [], [], []
                for c in range(5):
                    t_p = os.path.join(base_directory, "Results", dataset, tm, "SaliencyMap", f"SaliencyText_{tm}_{(img_idx*5)+c}.npy")
                    if os.path.exists(t_p):
                        toks, scs = clean_and_normalize_text_data(np.load(t_p, allow_pickle=True).item())
                        for t, s in zip(toks, scs):
                            all_t.append(f"{t} (C{c+1})"); all_s.append(s); all_c.append(colors[c])
                
                if all_s:
                    idxs = np.argsort(all_s)[-20:]
                    axes[m_idx, 1].barh([all_t[i] for i in idxs], [all_s[i] for i in idxs], color=[all_c[i] for i in idxs])
            plt.savefig(os.path.join(output_dir, f"Consensus_{img_name}.png"))
            plt.close()

if __name__ == "__main__":
    PATH = "/home/aysel/tfe/TFE_Data"
    VM = ['resnet50', 'mobilenet_v3', 'vit', 'deit', 'clip_vision']
    TM = ['bert', 'roberta', 'rnn', 'gpt2', 'clip_text']
    

    generate_full_5_caption_report(PATH, ['Flickr8k', 'Flickr30k'], VM, TM, 5)
    generate_consensus_report(PATH, ['Flickr8k', 'Flickr30k'], TM, 5)